# Analiza hantavirusa primenom metoda istraživanja podataka
## Tema 1, Zadatak 1.2 — Klasterovanje nad karakteristikama ponavljajućih sekvenci

Za četiri vrste hantavirusa (Hantaan, Dobrava-Belgrade, Puumala, Sin Nombre) analiziraju se
ponavljajuće sekvence (repeats) u aminokiselinskim sekvencama GPC (glycoprotein precursor) i N
(nucleocapsid protein), sa ciljem primene klasterovanja nad izvedenim karakteristikama i
poređenja dobijenih klastera sa stvarnom taksonomskom pripadnošću vrsti.

# 1. Uvod i konfiguracija

Učitavanje biblioteka, definisanje putanja i mapiranje preuzetih FASTA fajlova
(NCBI Virus baza) na (vrsta, kompletnost).

In [57]:
import re
import os
import subprocess
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio import SeqIO

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

DATA_RAW = Path("data/raw")
DATA_FILTERED = Path("data/filtered")
REPEAT_OUTPUT_DIR = Path("repeat_output")
FIGURES_DIR = Path("figures")

DATA_FILTERED.mkdir(parents=True, exist_ok=True)
REPEAT_OUTPUT_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)


# 1. Uvod i konfiguracija (nastavak)

Mapiranje preuzetih FASTA fajlova (NCBI Virus baza) na (vrsta, kompletnost), na osnovu
broja sekvenci po fajlu i sadržaja headera (potvrđeno ranijom inspekcijom).

In [58]:
FILE_SPECIES_MAP = {
    "hantaan_all.fasta": ("hantanense", "all"),
    "hantaan_complete.fasta": ("hantanense", "complete"),
    "dobrava_all.fasta": ("dobravaense", "all"),
    "dobrava_complete.fasta": ("dobravaense", "complete"),
    "puumala_all.fasta": ("puumalaense", "all"),
    "puumala_complete.fasta": ("puumalaense", "complete"),
    "sinnombre_all.fasta": ("sinnombreense", "all"),
    "sinnombre_complete.fasta": ("sinnombreense", "complete"),
}

for fname in FILE_SPECIES_MAP:
    path = DATA_RAW / fname
    assert path.exists(), f"Fajl ne postoji: {path}"
print("Svi fajlovi pronađeni.")


Svi fajlovi pronađeni.
